In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Qwen2.5-72B-Instruct project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [3]:
# pip install -U bitsandbytes
!pip install torch


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-72B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
#     load_in_8bit=True,  # This will solve your OOM issue
#     max_memory={i: "45GB" for i in range(8)},  # Limit per GPU
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    temperature=0.0,
    do_sample=False
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|█████████████████████████████| 37/37 [02:41<00:00,  4.37s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [2]:
import pandas as pd 
import re
import torch
import os

# Load data
df = pd.read_csv(paths.DATA / "Centaur_Lab_First_Round_COMPLETE_RAW.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None


# At the beginning, before the loop
progress_file = paths.PREDICTIONS / "Qwen_72B_predictions_on_Trainee_progress.csv"

# Check if progress file exists and load it
if os.path.exists(progress_file):
    existing_results = pd.read_csv(progress_file)
    processed_indices = set(existing_results.index.values)
    results = existing_results.to_dict('records')
    print(f"Found {len(processed_indices)} already processed rows. Resuming...")
else:
    processed_indices = set()
    results = []
    print("Starting from scratch...")

    
# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    # Skip if already processed
    if idx in processed_indices:
        print(f"Skipping row {idx+1}/{total_rows} (already processed)...")
        continue
        
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["New_Sentences"]
        question = row["question_options"]
        
        # Improved prompt with clearer instructions
        query_full = (
            "You are a clinical reasoning assistant. You will receive a patient case summary "
            "and a multiple-choice question.\n\n"
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Please select the single most appropriate answer. Respond only in the following format:\n\n"
            "Answer: <LETTER>"
        )
    
        # Generate prediction using the model
        inputs = tokenizer(query_full, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode the generated response
        raw_response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # Extract the answer letter using improved function
        extracted_answer = extract_answer_letter(raw_response)
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
        qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        processed_indices.add(idx)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(progress_file, index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })
        processed_indices.add(idx)

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "Qwen_72B_predictions_Trainee.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")

Columns in dataset:
['Origin', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'ID_corr', 'sentence_number_corr', 'answer_corr', 'data_source_corr', 'REMOVED_Sentences', 'sentence_number_df3', 'step1_excerpts', 'question_options', 'Filtered_Sentences', 'New_Sentences']
Found 760 already processed rows. Resuming...
Processing 1300 rows...
Skipping row 1/1300 (already processed)...
Skipping row 2/1300 (already processed)...
Skipping row 3/1300 (already processed)...
Skipping row 4/1300 (already processed)...
Skipping row 5/1300 (already processed)...
Skipping row 6/1300 (already processed)...
Skipping row 7/1300 (already processed)...
Skipping row 8/1300 (already processed)...
Skipping row 9/1300 (already processed)...
Skipping row 10/1300 (already processed)...
Skipping row 11/1300 (already processed)...
Skipping row 12/1300 (already processed)...
Skipping row 13/1300 (already processed)...
Skipping row 1

✅ Processed Merge Q761: Answer = C
Processing row 762/1300...
✅ Processed Merge Q762: Answer = C
Processing row 763/1300...
✅ Processed Merge Q763: Answer = A
Processing row 764/1300...
✅ Processed Merge Q764: Answer = C
Processing row 765/1300...
✅ Processed Merge Q765: Answer = B
Processing row 766/1300...
✅ Processed Merge Q766: Answer = D
Processing row 767/1300...
✅ Processed Merge Q767: Answer = B
Processing row 768/1300...
✅ Processed Merge Q768: Answer = D
Processing row 769/1300...
⚠️ Could not extract answer from response for row 769:
Response: To correct the metabolic abnormalities associated with tumor lysis syndrome (TLS) in this patient wh...
✅ Processed Merge Q769: Answer = None
Processing row 770/1300...
✅ Processed Merge Q770: Answer = B
Saved progress to CSV after 770 items
Processing row 771/1300...
✅ Processed Merge Q771: Answer = B
Processing row 772/1300...
✅ Processed Merge Q772: Answer = B
Processing row 773/1300...
✅ Processed Merge Q773: Answer = D
Processing 

✅ Processed Merge Q852: Answer = A
Processing row 853/1300...
✅ Processed Merge Q853: Answer = D
Processing row 854/1300...
⚠️ Could not extract answer from response for row 854:
Response: To determine the correct answer, let's break down the information provided:

1. **Hemophilia Inherit...
✅ Processed Merge Q854: Answer = None
Processing row 855/1300...
✅ Processed Merge Q855: Answer = B
Processing row 856/1300...
⚠️ Could not extract answer from response for row 856:
Response: To address the patient's condition effectively, we need to consider the clinical findings and the po...
✅ Processed Merge Q856: Answer = None
Processing row 857/1300...
✅ Processed Merge Q857: Answer = G
Processing row 858/1300...
✅ Processed Merge Q858: Answer = C
Processing row 859/1300...
✅ Processed Merge Q859: Answer = D
Processing row 860/1300...
⚠️ Could not extract answer from response for row 860:
Response: To provide a rationale, consider the following points from the case:
- The presence of a grade 

✅ Processed Merge Q951: Answer = B
Processing row 952/1300...
✅ Processed Merge Q952: Answer = C
Processing row 953/1300...
✅ Processed Merge Q953: Answer = B
Processing row 954/1300...
⚠️ Could not extract answer from response for row 954:
Response: To ensure I provide the most accurate response, let's review the key points from the patient's case:...
✅ Processed Merge Q954: Answer = None
Processing row 955/1300...
✅ Processed Merge Q955: Answer = D
Processing row 956/1300...
✅ Processed Merge Q956: Answer = D
Processing row 957/1300...
✅ Processed Merge Q957: Answer = A
Processing row 958/1300...
✅ Processed Merge Q958: Answer = E
Processing row 959/1300...
✅ Processed Merge Q959: Answer = C
Processing row 960/1300...
✅ Processed Merge Q960: Answer = A
Saved progress to CSV after 960 items
Processing row 961/1300...
✅ Processed Merge Q961: Answer = D
Processing row 962/1300...
✅ Processed Merge Q962: Answer = A
Processing row 963/1300...
✅ Processed Merge Q963: Answer = A
Processing 

⚠️ Could not extract answer from response for row 1041:
Response: To ensure you receive the correct response, please confirm if you agree with the following summary o...
✅ Processed Merge Q1041: Answer = None
Processing row 1042/1300...
✅ Processed Merge Q1042: Answer = D
Processing row 1043/1300...
✅ Processed Merge Q1043: Answer = F
Processing row 1044/1300...
✅ Processed Merge Q1044: Answer = C
Processing row 1045/1300...
✅ Processed Merge Q1045: Answer = C
Processing row 1046/1300...
⚠️ Could not extract answer from response for row 1046:
Response: To provide a detailed explanation for my answer, I would need to consider the following points:
- Th...
✅ Processed Merge Q1046: Answer = None
Processing row 1047/1300...
✅ Processed Merge Q1047: Answer = H
Processing row 1048/1300...
⚠️ Could not extract answer from response for row 1048:
Response: To provide a detailed explanation, please ask me to elaborate.
To determine the correct answer, we n...
✅ Processed Merge Q1048: Answer = No

⚠️ Could not extract answer from response for row 1132:
Response: To respond to this case, I will consider the clinical presentation and the available options.

The p...
✅ Processed Merge Q1132: Answer = None
Processing row 1133/1300...
✅ Processed Merge Q1133: Answer = D
Processing row 1134/1300...
✅ Processed Merge Q1134: Answer = D
Processing row 1135/1300...
✅ Processed Merge Q1135: Answer = D
Processing row 1136/1300...
✅ Processed Merge Q1136: Answer = D
Processing row 1137/1300...
✅ Processed Merge Q1137: Answer = C
Processing row 1138/1300...
⚠️ Could not extract answer from response for row 1138:
Response: To ensure I provide the most accurate response, I need to consider the clinical presentation and the...
✅ Processed Merge Q1138: Answer = None
Processing row 1139/1300...
✅ Processed Merge Q1139: Answer = E
Processing row 1140/1300...
✅ Processed Merge Q1140: Answer = B
Saved progress to CSV after 1140 items
Processing row 1141/1300...
✅ Processed Merge Q1141: Answer = B
Pro

✅ Processed Merge Q1225: Answer = C
Processing row 1226/1300...
✅ Processed Merge Q1226: Answer = B
Processing row 1227/1300...
✅ Processed Merge Q1227: Answer = G
Processing row 1228/1300...
✅ Processed Merge Q1228: Answer = C
Processing row 1229/1300...
✅ Processed Merge Q1229: Answer = D
Processing row 1230/1300...
✅ Processed Merge Q1230: Answer = C
Saved progress to CSV after 1230 items
Processing row 1231/1300...
⚠️ Could not extract answer from response for row 1231:
Response: To address the condition described, it is crucial to identify the underlying cause of the retinal va...
✅ Processed Merge Q1231: Answer = None
Processing row 1232/1300...
✅ Processed Merge Q1232: Answer = B
Processing row 1233/1300...
✅ Processed Merge Q1233: Answer = D
Processing row 1234/1300...
✅ Processed Merge Q1234: Answer = F
Processing row 1235/1300...
✅ Processed Merge Q1235: Answer = C
Processing row 1236/1300...
✅ Processed Merge Q1236: Answer = C
Processing row 1237/1300...
✅ Processed Merge Q1

In [6]:
output_df

,QA_ID,Origin,data_source,Raw_Response,Extracted_Answer
0,Merge Q1,ID0002,NaN,ERROR: name 'qa_id' is not defined,NaN
1,Merge Q2,ID0003,NaN,"To culture the most likely causative organism,...",NaN
2,Merge Q2,ID0007,NaN,To further investigate the cause of the patien...,A
3,Merge Q2,ID0009,NaN,"To provide a clear and concise response, I wil...",D
4,Merge Q2,ID0010,NaN,"To properly transport an amputated finger, it ...",H
...,...,...,...,...,...
1295,Merge Q1296,ID1995,NaN,To address the symptoms of this 17-year-old gi...,D
1296,Merge Q1297,ID1996,NaN,"To provide a detailed explanation, please ask ...",D
1297,Merge Q1298,ID1997,NaN,"To ensure you receive the best care, we may ne...",C
1298,Merge Q1299,ID1998,NaN,"To provide a proper answer, I need to know wha...",NaN


In [7]:
import pandas as pd
import re
import numpy as np
from scipy import stats

# Load the model predictions
output_df = pd.read_csv(paths.PREDICTIONS / "Qwen_72B_predictions_Trainee.csv")
df = pd.read_csv(paths.DATA / "Centaur_Lab_First_Round_COMPLETE_RAW.csv")

# Merge the two dataframes on ID_corr
merged_df = pd.merge(df, output_df[['Origin', 'Extracted_Answer']], on='Origin', how='inner')

# Compare answers
merged_df['72B_on_Trainee_Match'] = merged_df['answer_corr'] == merged_df['Extracted_Answer']
merged_df['72B_on_Trainee_Match'] = merged_df['72B_on_Trainee_Match'].map({True: 'TRUE', False: 'FALSE'})

# Exact match as 0/1 for statistics
merged_df['match'] = (merged_df['answer_corr'] == merged_df['Extracted_Answer']).astype(int)

# Overall accuracy statistics
accuracy = merged_df['match'].mean()
std_dev = merged_df['match'].std()
n = len(merged_df)
se = std_dev / np.sqrt(n)
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Category-wise analysis (if data_source_corr exists in df)
if 'data_source_corr' in df.columns:
    merged_df['data_source_corr'] = merged_df['data_source_corr']
    category_stats = merged_df.groupby('data_source_corr')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()

    # Calculate 95% CI per category
    ci_lower, ci_upper = [], []
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100

    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()

# Summary table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

OVERALL ACCURACY ANALYSIS
Accuracy: 0.5369 (53.69%)
Standard Deviation: 0.4988
95% Confidence Interval: [0.5098, 0.5641]
95% CI (percentage): [50.98%, 56.41%]
Sample Size: 1300

CATEGORY-WISE ACCURACY ANALYSIS
data_source_corr  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
            jama    582       0.601375 0.490037 0.020313     0.561479     0.641270        60.137457  49.003653      56.147936      64.126978
      medbullets    207       0.628019 0.484505 0.033675     0.561627     0.694412        62.801932  48.450499      56.162665      69.441200
        medxpert    318       0.223270 0.417094 0.023389     0.177252     0.269289        22.327044  41.709450      17.725217      26.928871
            mmlu    193       0.761658 0.427177 0.030749     0.701009     0.822307        76.165803  42.717745      70.100900      82.230706


SUMMARY TABLE
            Metric           Value
  Overall Accuracy 0.5369 (53.69%)

In [3]:
import pandas as pd
import re
import numpy as np

# Load the model predictions
output_path = paths.PREDICTIONS / "Qwen_72B_predictions_Trainee.csv"
df = pd.read_csv(output_path)

# Extract the predicted letter from the format <answer>Option A</answer>
def extract_letter_from_xml(pred):
    if isinstance(pred, str):
        match = re.search(r"Option\s+([A-J])", pred, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None

df["gpt_letter"] = df["gpt5_direct_prediction"].apply(extract_letter_from_xml)

# Clean and standardize the ground truth answer
df["answer_letter"] = df["answer_corr"].astype(str).str.strip().str.upper()

# Compare predictions to ground truth
df["gpt_letter_match"] = df.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_letter"] else "Incorrect",
    axis=1
)

# Convert to binary for std calculation
df["gpt_letter_binary"] = df["gpt_letter_match"].map({"Correct": 1, "Incorrect": 0})

# Compute overall accuracy
correct_count = df["gpt_letter_binary"].sum()
total_count = df["gpt_letter_binary"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0

print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")

# Per-data source accuracy and std
for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = source_df["gpt_letter_binary"].sum()
    total = source_df["gpt_letter_binary"].notna().sum()
    acc = correct / total if total > 0 else 0
    std = source_df["gpt_letter_binary"].std(ddof=1) if total > 1 else float("nan")
    
    print(f"Data Source: {source}")
    print(f"  Correct Predictions: {correct}")
    print(f"  Total Predictions: {total}")
    print(f"  Accuracy: {acc:.2%}")
    print(f"  Std Dev: {std:.4f}\n")


KeyError: 'gpt5_direct_prediction'

In [ ]:
import numpy as np

# Collect accuracies per data source
accuracies = []

for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = (source_df["gpt_letter_match"] == "Correct").sum()
    total = source_df["gpt_letter_match"].notna().sum()
    acc = correct / total if total > 0 else 0
    accuracies.append(acc)

# Calculate standard deviation
accuracy_std = np.std(accuracies, ddof=1)  # use ddof=1 for sample std deviation
print(f"Standard Deviation of Accuracy Across Data Sources: {accuracy_std:.4f}")